In [45]:
import pandas as pd
import numpy as np
import re
import pickle
import os
import nltk
from nltk.stem import PorterStemmer

# Khai báo thu muc
RAW_DIR = "../data/raw"
PREPROCESSED_DIR = "../data/processed"
MODELS = "../models"
DICT_DIR = "../data/dicts"
for d in [RAW_DIR, MODELS, PREPROCESSED_DIR, DICT_DIR]:
    os.makedirs(d, exist_ok=True)

# Khai bao duong dan file
CORPUS_PATH = f"{RAW_DIR}/corpus.parquet"
QUERIES_PATH = f"{RAW_DIR}/queries.parquet"
QRELS_PATH = f"{RAW_DIR}/qrels.parquet"

In [46]:
# ── Corpus ───────────────────────────────────────────────────
# Đọc file bằng read_parquet
corpus_df = pd.read_parquet(CORPUS_PATH)

corpus_df = corpus_df.rename(columns={
    '_id':  'id',
    'text': 'abstract'
})
corpus_df['text'] = corpus_df['title'].fillna('') + ' ' + \
                    corpus_df['title'].fillna('') + ' ' + \
                    corpus_df['abstract'].fillna('')
corpus_df = corpus_df.dropna(subset=['text']).reset_index(drop=True)

print(f"Số papers: {len(corpus_df):,}")
print(f"Columns:   {list(corpus_df.columns)}")
print(corpus_df[['id','title','abstract']].head(3))

queries_df = pd.read_parquet(QUERIES_PATH)
queries_df = queries_df.rename(columns={'_id': 'query_id',
                                        'text': 'query_text'})

qrels_df = pd.read_parquet(QRELS_PATH)
qrels_df  = qrels_df.rename(columns={
    'query-id':  'query_id',
    'corpus-id': 'doc_id',
    'score':     'relevance'
})
qrels_df['query_id'] = qrels_df['query_id'].astype(str)
qrels_df['doc_id'] = qrels_df['doc_id'].astype(str)
qrels_df['relevance'] = qrels_df['relevance'].astype(int)

# 2. Thay thế iterrows() bằng groupby() kết hợp dictionary comprehension siêu tốc
qrels_dict = {
    qid: dict(zip(group['doc_id'], group['relevance']))
    for qid, group in qrels_df.groupby('query_id')
}

queries_dict = dict(zip(
    queries_df['query_id'].astype(str),
    queries_df['query_text']
))
# Lưu Dictionaries
pickle.dump(qrels_dict,   open(f'{DICT_DIR}/qrels_dict.pkl','wb'))
pickle.dump(queries_dict, open(f'{DICT_DIR}/queries_dict.pkl','wb'))

# Lưu lại Corpus cuối cùng cũng bằng định dạng parquet
corpus_df.to_parquet(f'{PREPROCESSED_DIR}/scidocs_corpus.parquet', index=False)

print("\nĐã tạo và lưu:")
print(f"  - scidocs_corpus.parquet ({len(corpus_df):,} papers)")
print(f"  - qrels_dict.pkl ({len(qrels_dict):,} queries)")
print(f"  - queries_dict.pkl ({len(queries_dict):,} queries)")

Số papers: 25,657
Columns:   ['id', 'title', 'abstract', 'text']
                                         id  \
0  632589828c8b9fca2c3a59e97451fde8fa7d188d   
1  86e87db2dab958f1bd5877dc7d5b8105d6e31e46   
2  2a047d8c4c2a4825e0f0305294e7da14f8de6fd3   

                                               title  \
0  A hybrid of genetic algorithm and particle swa...   
1  A Hybrid EP and SQP for Dynamic Economic Dispa...   
2  Genetic Fuzzy Systems - Evolutionary Tuning an...   

                                            abstract  
0  An evolutionary recurrent network which automa...  
1  Dynamic economic dispatch (DED) is one of the ...  
2  It's not surprisingly when entering this site ...  

Đã tạo và lưu:
  - scidocs_corpus.parquet (25,657 papers)
  - qrels_dict.pkl (1,000 queries)
  - queries_dict.pkl (1,000 queries)


In [47]:
nltk.download('stopwords', quiet=True)

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

from tqdm.auto import tqdm
tqdm.pandas(desc="Đang xử lý")


corpus_df = pd.read_parquet(f'{PREPROCESSED_DIR}/scidocs_corpus.parquet')

# Loại bỏ các dòng trống và reset index
corpus_df = corpus_df.dropna(subset=['text']).reset_index(drop=True)

print(f"Shape: {corpus_df.shape}")
print(corpus_df[['id','title','abstract']].head(3))

Shape: (25657, 4)
                                         id  \
0  632589828c8b9fca2c3a59e97451fde8fa7d188d   
1  86e87db2dab958f1bd5877dc7d5b8105d6e31e46   
2  2a047d8c4c2a4825e0f0305294e7da14f8de6fd3   

                                               title  \
0  A hybrid of genetic algorithm and particle swa...   
1  A Hybrid EP and SQP for Dynamic Economic Dispa...   
2  Genetic Fuzzy Systems - Evolutionary Tuning an...   

                                            abstract  
0  An evolutionary recurrent network which automa...  
1  Dynamic economic dispatch (DED) is one of the ...  
2  It's not surprisingly when entering this site ...  


In [48]:
# 1. Stopwords tiếng Anh chuẩn 
STANDARD_STOP = set(stopwords.words('english'))

# 2. Stopwords chuyên ngành đã được cắt gốc 
CUSTOM_STOP_STEMMED = {
    'paper', 'propos', 'method', 'approach',
    'result', 'show', 'base', 'also', 'howev',
    'therefor', 'thu', 'furthermor', 'present',
    'work', 'studi', 'experiment', 'evalu',
    'use', 'new', 'high'
}

stemmer = PorterStemmer()

def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'http\S+',  '', text)
    text = re.sub(r'\$.*?\$',  ' math ', text)  # Thay thế công thức LaTeX
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+',      ' ', text).strip()
    return text

def remove_standard_stop(tokens):
    return [t for t in tokens if t not in STANDARD_STOP and len(t) > 2]

# ── TV1: TF-IDF + Boolean (Stemming) ─────────────────────────
def preprocess_tfidf(text: str) -> str:
    tokens = remove_standard_stop(clean_text(text).split())
    # Cắt gốc xong mới kiểm tra với tập CUSTOM_STOP_STEMMED
    stemmed_tokens = [stemmer.stem(t) for t in tokens]
    final_tokens = [t for t in stemmed_tokens if t not in CUSTOM_STOP_STEMMED]
    return ' '.join(final_tokens)

sample = corpus_df['text'].iloc[0]
print("ORIGINAL:\n",    sample[:200])
print("\nTF-IDF:\n",    preprocess_tfidf(sample)[:200])

ORIGINAL:
 A hybrid of genetic algorithm and particle swarm optimization for recurrent network design A hybrid of genetic algorithm and particle swarm optimization for recurrent network design A hybrid of geneti

TF-IDF:
 hybrid genet algorithm particl swarm optim recurr network design hybrid genet algorithm particl swarm optim recurr network design hybrid genet algorithm particl swarm optim recurr network design evolu


In [49]:
print("Tiền xử lý TV1 — TF-IDF / Boolean")
corpus_df['text_tfidf'] = corpus_df['text'].progress_apply(preprocess_tfidf)


print(corpus_df[['title','text_tfidf']].head(2))

Tiền xử lý TV1 — TF-IDF / Boolean


Đang xử lý:   0%|          | 0/25657 [00:00<?, ?it/s]

                                               title  \
0  A hybrid of genetic algorithm and particle swa...   
1  A Hybrid EP and SQP for Dynamic Economic Dispa...   

                                          text_tfidf  
0  hybrid genet algorithm particl swarm optim rec...  
1  hybrid sqp dynam econom dispatch nonsmooth fue...  


In [50]:
from collections import Counter


words = ' '.join(corpus_df['text_tfidf']).split()
freq  = Counter(words)


print(f"  Tổng số Tokens: {len(words):,}")
print(f"  Số từ vựng (Vocab):  {len(freq):,}")
print(f"  Top 15 từ phổ biến: {[w for w,_ in freq.most_common(15)]}")

  Tổng số Tokens: 3,025,925
  Số từ vựng (Vocab):  55,678
  Top 15 từ phổ biến: ['model', 'system', 'data', 'network', 'learn', 'perform', 'algorithm', 'imag', 'gener', 'design', 'applic', 'time', 'inform', 'detect', 'comput']


In [51]:
import os

OUTPUT_FILE = 'data_tv1.parquet'

# Lưu file riêng cho thành viên 1
corpus_df[['id', 'title', 'abstract', 'text_tfidf']]\
    .to_parquet(f'{PREPROCESSED_DIR}/{OUTPUT_FILE}')

# Kiểm tra dung lượng file
file_path = f'{PREPROCESSED_DIR}/{OUTPUT_FILE}'
if os.path.exists(file_path):
    size = os.path.getsize(file_path) / 1024 / 1024
    print(f"Đã lưu thành công!")
    print(f"  {OUTPUT_FILE:<20} {size:.2f} MB")

Đã lưu thành công!
  data_tfidf-boolean.parquet 27.59 MB
